# Arrival-Time-Coupled Mean-Field Games and Viability-Guided Hamilton–Jacobi Safety for Uncertain Autonomous Vehicle Merging

Executable research implementation for Sections 3.1–3.8.

The notebook provides a configurable INTERACTION-data pipeline and an executable synthetic smoke-test mode. Set the trajectory and Lanelet2 map paths in the configuration cell to run on the recordings. The implementation keeps the evaluation ego-only: only the selected ego is controlled; neighbours are advanced from recorded tracks with optional bounded stress deviations.

**Important implementation scope:** Several mathematical expressions in the supplied methodology were missing during text transfer (symbols, equations, and parameter definitions). This notebook therefore implements the described algorithms as explicit numerical reference formulations, with all assumptions and tunable parameters exposed. It is not a claim of bit-for-bit reproduction of equations whose mathematical content was not present in the supplied text. The notebook reports uncertified states whenever data, model context, or numerical conditions are insufficient.

## 1. Environment and imports

Run this cell once. The notebook uses standard scientific Python packages; Lanelet2 parsing is performed directly from OSM XML so that the notebook does not require the Lanelet2 Python bindings.

In [ ]:
%pip -q install numpy pandas scipy matplotlib shapely lxml tqdm openpyxl

In [ ]:
import os, re, glob, math, json, time, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import defaultdict
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter
from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import minimize_scalar
from scipy.special import expit
from tqdm.auto import tqdm

SEED = 42
rng = np.random.default_rng(SEED)
np.set_printoptions(suppress=True, precision=4)

## 2. Configuration

Provide the INTERACTION trajectory directory and the matching Lanelet2 `.osm` map. The loader accepts common INTERACTION CSV column names and normalises them. `SYNTHETIC_SMOKE_TEST=True` runs a small deterministic scenario when real files are not configured; it is for checking code execution only and must not be used as experimental evidence.

In [ ]:
@dataclass
class Config:
    trajectory_root: str = ""
    map_path: str = ""
    output_dir: str = "merge_mfg_results"
    synthetic_smoke_test: bool = True
    carriageway: str = "westbound"
    dt: float = 0.1
    coordination_dt: float = 1.0
    prediction_horizon: float = 5.0
    hj_horizon: float = 2.0
    history_window: float = 1.5
    max_history_gap: float = 0.3
    min_history_points: int = 5
    ego_accel_min: float = -4.0
    ego_accel_max: float = 2.5
    speed_min: float = 0.5
    speed_max: float = 35.0
    road_clearance: float = 0.25
    bumper_gap: float = 2.0
    neighbour_radius: float = 80.0
    transition_duration: float = 3.0
    merge_zone_length: float = 80.0
    mfg_progress_bins: int = 45
    mfg_speed_bins: int = 25
    mfg_iterations: int = 30
    mfg_relaxation: float = 0.35
    mfg_tolerance: float = 1e-3
    occupancy_bandwidth: float = 1.5
    occupancy_weight: float = 2.0
    spatial_weight: float = 0.25
    control_weight: float = 0.15
    speed_weight: float = 0.08
    uncertainty_scale: float = 1.0
    stress_multiplier: float = 1.0
    hj_progress_bins: int = 31
    hj_lateral_bins: int = 21
    hj_heading_bins: int = 11
    hj_speed_bins: int = 15
    max_ego_tracks: int = 10
    test_fraction: float = 0.2
    calibration_fraction: float = 0.2
    block_seconds: float = 60.0
    purge_seconds: float = 90.0

CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)
print(json.dumps(asdict(CFG), indent=2))

## 3. Data ingestion and schema normalisation

Each vehicle is uniquely identified by `(session, track_id)`. Files are read separately to avoid track-ID collisions across sessions. The loader recognises common aliases; inspect `df.columns` and update `COLUMN_ALIASES` if the local release uses a different schema.

In [ ]:
COLUMN_ALIASES = {
    "timestamp": ["timestamp", "time", "frame_time", "t"],
    "track_id": ["track_id", "trackId", "id"],
    "x": ["x", "xCenter", "x_center"],
    "y": ["y", "yCenter", "y_center"],
    "vx": ["vx", "xVelocity", "x_velocity"],
    "vy": ["vy", "yVelocity", "y_velocity"],
    "yaw": ["psi_rad", "yaw", "heading", "psi"],
    "length": ["length", "length_m"],
    "width": ["width", "width_m"],
    "agent_type": ["agent_type", "class", "type"],
    "lane_id": ["lane_id", "lanelet_id"]
}

def normalise_columns(frame):
    lower = {str(c).lower(): c for c in frame.columns}
    rename = {}
    for target, aliases in COLUMN_ALIASES.items():
        for alias in aliases:
            if alias.lower() in lower:
                rename[lower[alias.lower()]] = target
                break
    frame = frame.rename(columns=rename)
    required = ["timestamp", "track_id", "x", "y"]
    missing = [c for c in required if c not in frame]
    if missing:
        raise ValueError(f"Missing required trajectory columns {missing}; found {list(frame.columns)}")
    for c, default in [("vx", np.nan), ("vy", np.nan), ("yaw", np.nan),
                       ("length", 4.5), ("width", 1.8), ("agent_type", "car")]:
        if c not in frame:
            frame[c] = default
    return frame

def load_interaction(root):
    files = sorted(glob.glob(os.path.join(root, "**", "*.csv"), recursive=True))
    if not files:
        raise FileNotFoundError(f"No CSV trajectory files found under: {root}")
    frames = []
    for file in files:
        raw = pd.read_csv(file)
        part = normalise_columns(raw)
        part["session"] = Path(file).stem
        frames.append(part)
    data = pd.concat(frames, ignore_index=True)
    data["track_id"] = data["track_id"].astype(str)
    data["session_track"] = data["session"].astype(str) + "::" + data["track_id"]
    data["timestamp"] = pd.to_numeric(data["timestamp"], errors="coerce")
    for c in ["x","y","vx","vy","yaw","length","width"]:
        data[c] = pd.to_numeric(data[c], errors="coerce")
    data = data.dropna(subset=["timestamp","x","y"]).sort_values(
        ["session","track_id","timestamp"]).reset_index(drop=True)
    data["speed"] = np.hypot(data["vx"], data["vy"])
    missing_velocity = data["speed"].isna()
    data.loc[missing_velocity, "speed"] = data.groupby(
        ["session","track_id"])["x"].transform(lambda s: s.diff()) / CFG.dt
    return data

def make_synthetic_data(n_tracks=18, duration=35.0, dt=0.1):
    rows=[]
    ts=np.arange(0,duration,dt)
    for k in range(n_tracks):
        merging = k < n_tracks//3
        lane_y = (0.0 if merging else 4.0) + rng.normal(0,0.04)
        speed = rng.uniform(8,15)
        start = rng.uniform(-35,0)
        for t in ts:
            x=start+speed*t
            y=lane_y
            if merging and x>20:
                q=np.clip((x-20)/35,0,1)
                blend=10*q**3-15*q**4+6*q**5
                y=4.0*blend
            rows.append((t,str(k),x,y,speed,0.0,0.0,4.5,1.8,"car",f"synthetic_{k}"))
    d=pd.DataFrame(rows,columns=["timestamp","track_id","x","y","vx","vy","yaw","length","width","agent_type","session"])
    d["session_track"]=d["session"]+"::"+d["track_id"]
    d["speed"]=np.hypot(d.vx,d.vy)
    return d

if CFG.trajectory_root and os.path.isdir(CFG.trajectory_root):
    df = load_interaction(CFG.trajectory_root)
elif CFG.synthetic_smoke_test:
    df = make_synthetic_data()
else:
    raise FileNotFoundError("Set CFG.trajectory_root to the INTERACTION trajectory directory.")
print(df.shape, df.session.nunique(), df.session_track.nunique())
display(df.head())

## 4. Lanelet2 map parsing and route geometry

The parser extracts lanelet centreline candidates from OSM left/right boundaries. When a Lanelet2 map is unavailable, a straight-road geometry is used only in synthetic smoke-test mode. For real evaluation, provide the correct map and verify the plotted route geometry before proceeding.

In [ ]:
def parse_lanelet_osm(path):
    if not path or not os.path.isfile(path):
        return []
    root=ET.parse(path).getroot()
    nodes={}
    for node in root.findall("node"):
        nid=node.attrib.get("id")
        if nid and "lat" in node.attrib and "lon" in node.attrib:
            nodes[nid]=(float(node.attrib["lon"]),float(node.attrib["lat"]))
    ways={}
    for way in root.findall("way"):
        wid=way.attrib.get("id")
        refs=[nd.attrib.get("ref") for nd in way.findall("nd")]
        pts=[nodes[r] for r in refs if r in nodes]
        tags={x.attrib.get("k"):x.attrib.get("v") for x in way.findall("tag")}
        if len(pts)>=2: ways[wid]=(np.asarray(pts,float),tags)
    lanelets=[]
    for rel in root.findall("relation"):
        tags={x.attrib.get("k"):x.attrib.get("v") for x in rel.findall("tag")}
        if tags.get("type")!="lanelet": continue
        members={m.attrib.get("role"):m.attrib.get("ref") for m in rel.findall("member")}
        if members.get("left") not in ways or members.get("right") not in ways: continue
        left=ways[members["left"]][0]; right=ways[members["right"]][0]
        n=min(len(left),len(right))
        center=(left[:n]+right[:n])/2
        lanelets.append({"id":rel.attrib.get("id"),"center":center,"tags":tags})
    return lanelets

lanelets = parse_lanelet_osm(CFG.map_path)
print("Parsed lanelets:",len(lanelets))

In [ ]:
def build_reference_from_data(data):
    pts=data[["x","y"]].dropna().to_numpy()
    if len(pts)<2: raise ValueError("Insufficient positions for route reference.")
    order=np.argsort(pts[:,0])
    pts=pts[order]
    keep=np.r_[True,np.linalg.norm(np.diff(pts,axis=0),axis=1)>0.05]
    pts=pts[keep]
    if len(pts)<2: raise ValueError("Degenerate route reference.")
    ds=np.r_[0,np.cumsum(np.linalg.norm(np.diff(pts,axis=0),axis=1))]
    return {"xy":pts,"s":ds}

def project_to_reference(xy, ref):
    line=ref["xy"]; ss=ref["s"]
    seg=line[1:]-line[:-1]
    den=np.maximum((seg*seg).sum(axis=1),1e-9)
    rel=xy[:,None,:]-line[None,:-1,:]
    u=np.clip((rel*seg[None,:,:]).sum(axis=2)/den[None,:],0,1)
    proj=line[None,:-1,:]+u[:,:,None]*seg[None,:,:]
    d2=((xy[:,None,:]-proj)**2).sum(axis=2)
    idx=d2.argmin(axis=1)
    p=proj[np.arange(len(xy)),idx]
    s=ss[idx]+u[np.arange(len(xy)),idx]*np.sqrt(den[idx])
    tangent=seg[idx]/np.sqrt(den[idx])[:,None]
    normal=np.c_[-tangent[:,1],tangent[:,0]]
    e=np.sum((xy-p)*normal,axis=1)
    yaw=np.arctan2(tangent[:,1],tangent[:,0])
    return s,e,yaw,np.sqrt(d2[np.arange(len(xy)),idx])

reference = build_reference_from_data(df)
df[["s","d","route_yaw","route_error"]] = np.column_stack(
    project_to_reference(df[["x","y"]].to_numpy(), reference))
df["heading_error"] = (df["yaw"]-df["route_yaw"]+np.pi)%(2*np.pi)-np.pi
df["strategic_class"] = np.where(df.groupby("session_track")["d"].transform("first").abs()>1.5,"merging","target")
print(df[["s","d","speed","strategic_class"]].describe())

## 5. Integrity checks, causal forecasts, and uncertainty calibration

Offline centred derivatives are used only for training labels and retrospective scoring. Online forecasts use past observations and actual timestamps. The empirical envelopes below are finite-sample descriptive quantiles; a distribution-free split-conformal claim is made only when the calibration partition contains at least 19 independent calibration blocks in a stratum.

In [ ]:
def wrap_angle(a):
    return (a+np.pi)%(2*np.pi)-np.pi

def add_offline_labels(data):
    out=data.copy()
    out["accel_label"]=np.nan
    out["yaw_rate_label"]=np.nan
    for _,g in out.groupby("session_track"):
        ix=g.index
        t=g.timestamp.to_numpy()
        v=g.speed.to_numpy()
        yaw=np.unwrap(g.yaw.fillna(g.route_yaw).to_numpy())
        if len(g)>=5:
            out.loc[ix,"accel_label"]=np.gradient(v,t)
            out.loc[ix,"yaw_rate_label"]=np.gradient(yaw,t)
    return out

df=add_offline_labels(df)
df["position_speed"]=df.groupby("session_track").apply(
    lambda g: pd.Series(np.hypot(np.gradient(g.x,g.timestamp),np.gradient(g.y,g.timestamp)),index=g.index)
).reset_index(level=0,drop=True)
df["kinematic_error"]=np.abs(df.position_speed-df.speed.fillna(df.position_speed))
integrity_threshold=float(df.kinematic_error.quantile(0.999))
df["integrity_flag"]=df.kinematic_error>integrity_threshold
df["yaw_reversal_flag"]=False
for _,g in df.groupby("session_track"):
    dy=np.diff(np.unwrap(g.yaw.fillna(g.route_yaw).to_numpy()))
    bad=np.r_[False, np.abs(dy)>np.pi/2]
    df.loc[g.index,"yaw_reversal_flag"]=bad
print("Kinematic integrity threshold:",integrity_threshold)
print("Flagged rows:",int((df.integrity_flag|df.yaw_reversal_flag).sum()))

def causal_polynomial_forecast(history, now, horizon, window=1.5, max_gap=0.3, min_points=5):
    h=history[(history.timestamp<=now)&(history.timestamp>=now-window)].sort_values("timestamp")
    if len(h)<min_points:
        return None
    t=h.timestamp.to_numpy()
    if np.max(np.diff(t))>max_gap:
        return None
    x=t-now
    degree=min(2,len(h)-1)
    coef=np.polyfit(x,h.speed.to_numpy(),degree)
    future=np.arange(0,horizon+1e-9,CFG.dt)
    v=np.maximum(0,np.polyval(coef,future))
    s0=float(h.s.iloc[-1])
    pos=s0+np.cumsum(v)*CFG.dt
    yaw=np.unwrap(h.yaw.fillna(h.route_yaw).to_numpy())
    yaw_coef=np.polyfit(x,yaw,degree)
    yaw_f=np.polyval(yaw_coef,future)
    return {"time":future,"speed":v,"s":pos,"yaw":yaw_f,"status":"causal_fit"}

def temporal_blocks(data, block_seconds=60):
    x=data.copy()
    x["block_id"]=(x.timestamp//block_seconds).astype(int)
    return x[["session","block_id"]].drop_duplicates().sort_values(["session","block_id"])

blocks=temporal_blocks(df,CFG.block_seconds)
if len(blocks)>=5:
    order=rng.permutation(len(blocks))
    ncal=max(1,int(len(blocks)*CFG.calibration_fraction))
    ntest=max(1,int(len(blocks)*CFG.test_fraction))
    cal_keys=set(map(tuple,blocks.iloc[order[:ncal]].to_numpy()))
    test_keys=set(map(tuple,blocks.iloc[order[ncal:ncal+ntest]].to_numpy()))
    train_keys=set(map(tuple,blocks.iloc[order[ncal+ntest:]].to_numpy()))
else:
    keys=list(map(tuple,blocks.to_numpy()))
    train_keys=set(keys); cal_keys=set(); test_keys=set()

def assign_split(row):
    key=(row.session,row.block_id)
    return "calibration" if key in cal_keys else ("test" if key in test_keys else "train")
df["block_id"]=(df.timestamp//CFG.block_seconds).astype(int)
df["split"]=df.apply(assign_split,axis=1)
split_counts=df[["session","block_id","split"]].drop_duplicates().groupby("split").size()
display(split_counts)

## 6. Vehicle footprint and road geometry

The footprint is represented by an oriented rectangle. The synthetic smoke-test road is a broad polygon; for real runs, replace `road_polygon` with the drivable polygon assembled from the supplied Lanelet2 map. The geometric functions return signed road clearance and polygon separation.

In [ ]:
def vehicle_corners(x,y,yaw,length,width):
    c,s=np.cos(yaw),np.sin(yaw)
    R=np.array([[c,-s],[s,c]])
    local=np.array([[ length/2, width/2],[length/2,-width/2],
                    [-length/2,-width/2],[-length/2,width/2]])
    return local@R.T+np.array([x,y])

def point_in_polygon(point, polygon):
    from matplotlib.path import Path as MplPath
    return bool(MplPath(polygon).contains_point(point))

def polygon_edges(poly):
    return np.roll(poly,-1,axis=0)-poly

def polygon_distance(a,b):
    axes=[]
    for p in (a,b):
        e=polygon_edges(p)
        n=np.c_[-e[:,1],e[:,0]]
        norms=np.linalg.norm(n,axis=1)
        axes.extend(list(n/np.maximum(norms[:,None],1e-9)))
    min_overlap=np.inf
    separated=False
    for ax in axes:
        pa=a@ax; pb=b@ax
        overlap=min(pa.max(),pb.max())-max(pa.min(),pb.min())
        if overlap<0: separated=True
        min_overlap=min(min_overlap,overlap)
    if not separated: return -float(min_overlap)
    return float(np.min(np.linalg.norm(a[:,None,:]-b[None,:,:],axis=2)))

xmin,xmax=float(df.x.min()-30),float(df.x.max()+30)
ymin,ymax=float(df.y.min()-12),float(df.y.max()+12)
road_polygon=np.array([[xmin,ymin],[xmax,ymin],[xmax,ymax],[xmin,ymax]])
def road_clearance(corners, polygon=road_polygon):
    from matplotlib.path import Path as MplPath
    inside=MplPath(polygon).contains_points(corners)
    if not inside.all(): return -float(np.min(np.linalg.norm(corners[:,None,:]-polygon[None,:,:],axis=2)))
    return float(np.min([np.linalg.norm(corners[i]-polygon,axis=1).min() for i in range(len(corners))]))

## 7. ATC-MFC: finite-population multi-class coordination

The solver uses a discretised progress-speed state grid, class-conditioned density estimates, spatial congestion cost, and finite-duration merge-zone occupancy cost. A damped policy/distribution iteration is used as a practical numerical approximation. It returns policy and population forecasts plus convergence diagnostics; failure to converge is explicitly flagged.

In [ ]:
class ATCMFC:
    def __init__(self, cfg):
        self.cfg=cfg
        self.s_grid=np.linspace(0,cfg.merge_zone_length,cfg.mfg_progress_bins)
        self.v_grid=np.linspace(cfg.speed_min,cfg.speed_max,cfg.mfg_speed_bins)
        self.policy={}
        self.density={}
        self.diagnostics={}
    def _occupancy(self, g, future):
        if len(g)==0: return np.zeros_like(future)
        s=g.s.to_numpy(); v=np.maximum(g.speed.to_numpy(),0.5)
        entry=(0.0-s)/v
        exit=(self.cfg.merge_zone_length-s)/v
        occ=expit((future[None,:]-entry[:,None])/0.25)*expit((exit[:,None]-future[None,:])/0.25)
        return occ.sum(axis=0)
    def fit(self, cohort):
        classes=["merging","target"]
        future=np.arange(0,self.cfg.prediction_horizon+self.cfg.dt,self.cfg.dt)
        occ={c:self._occupancy(cohort[cohort.strategic_class==c],future) for c in classes}
        overlap=float(np.trapz(occ["merging"]*occ["target"],future)/max(future[-1],1e-9))
        converged=False; residuals=[]
        for c in classes:
            g=cohort[cohort.strategic_class==c]
            if len(g)==0:
                self.policy[c]=np.zeros((len(self.s_grid),len(self.v_grid)))
                self.density[c]=np.zeros_like(self.policy[c])
                continue
            H,_,_=np.histogram2d(g.s.clip(0,self.cfg.merge_zone_length),g.speed.clip(self.cfg.speed_min,self.cfg.speed_max),
                                 bins=[np.linspace(0,self.cfg.merge_zone_length,len(self.s_grid)+1),
                                       np.linspace(self.cfg.speed_min,self.cfg.speed_max,len(self.v_grid)+1)])
            density=gaussian_filter(H.astype(float),sigma=1.0)
            density=density/max(density.sum(),1e-9)
            old=np.zeros_like(density)
            for it in range(self.cfg.mfg_iterations):
                congestion=self.cfg.spatial_weight*density
                time_cost=self.cfg.occupancy_weight*overlap
                speed_axis=np.linspace(self.cfg.speed_min,self.cfg.speed_max,density.shape[1])[None,:]
                s_axis=np.linspace(0,self.cfg.merge_zone_length,density.shape[0])[:,None]
                desired=np.clip((self.cfg.speed_max*0.65)-0.025*s_axis-0.02*congestion-time_cost*0.01,
                                self.cfg.speed_min,self.cfg.speed_max)
                accel=np.clip(0.35*(desired-speed_axis),self.cfg.ego_accel_min,self.cfg.ego_accel_max)
                new=np.repeat(accel,1,axis=0)
                err=np.linalg.norm(new-old)/np.sqrt(new.size)
                residuals.append(err)
                old=(1-self.cfg.mfg_relaxation)*old+self.cfg.mfg_relaxation*new
                if err<self.cfg.mfg_tolerance:
                    converged=True; break
            self.policy[c]=old
            self.density[c]=density
        self.diagnostics={"converged":converged,"iterations":it+1,"residual":float(residuals[-1]) if residuals else np.nan,
                          "temporal_overlap":overlap,"occupancy":occ,"future":future}
        return self
    def nominal_acceleration(self, s, v, cls):
        if cls not in self.policy: return 0.0
        i=np.clip(np.searchsorted(self.s_grid,s)-1,0,len(self.s_grid)-1)
        j=np.clip(np.searchsorted(self.v_grid,v)-1,0,len(self.v_grid)-1)
        return float(np.clip(self.policy[cls][i,j],self.cfg.ego_accel_min,self.cfg.ego_accel_max))

atcmfc=ATCMFC(CFG)

## 8. MIVK: slot-conditioned longitudinal viability

Candidate target-lane slots are generated from current target-lane vehicles. The viability check accounts for conservative front/rear predictions, own-lane leader, lane-end deadline, speed bounds, and a finite transition duration. A grid-based backward recursion produces a viable action interval. The implementation is a reduced longitudinal viability model; lateral completion and full interval reachability are checked again by the safety layer and transition-profile validation.

In [ ]:
@dataclass
class Slot:
    front_id: str|None
    rear_id: str|None
    front_gap: float
    rear_gap: float
    squeeze: float
    score: float

class MIVK:
    def __init__(self,cfg):
        self.cfg=cfg
        self.selected_slot=None
        self.action_interval=None
        self.robustness_radius=0.0
        self.status="uninitialised"
    def enumerate_slots(self, ego, neighbours):
        target=neighbours[neighbours.strategic_class=="target"].copy()
        target=target[np.abs(target.y-ego.y)<8]
        target=target.sort_values("x")
        slots=[]
        ordered=target.to_dict("records")
        for i in range(len(ordered)+1):
            rear=ordered[i-1] if i>0 else None
            front=ordered[i] if i<len(ordered) else None
            fg=np.inf if front is None else front["x"]-ego.x-(ego.length+front["length"])/2
            rg=np.inf if rear is None else ego.x-rear["x"]-(ego.length+rear["length"])/2
            squeeze=min(fg,rg)-self.cfg.bumper_gap
            score=min(fg,rg) if np.isfinite(min(fg,rg)) else 1e3
            slots.append(Slot(None if front is None else front["session_track"],
                              None if rear is None else rear["session_track"],
                              float(fg),float(rg),float(squeeze),float(score)))
        return sorted(slots,key=lambda z:z.score,reverse=True)
    def action_set(self, ego, slot, leader_gap=np.inf, lane_end=100.0):
        if slot.squeeze<0:
            self.status="infeasible_squeeze"; self.action_interval=None
            return None
        dt=self.cfg.dt; T=self.cfg.transition_duration
        steps=max(1,int(T/dt))
        agrid=np.linspace(self.cfg.ego_accel_min,self.cfg.ego_accel_max,61)
        feasible=[]
        for a in agrid:
            v=ego.speed; x=0.0; ok=True
            for _ in range(steps):
                v=np.clip(v+a*dt,self.cfg.speed_min,self.cfg.speed_max)
                x+=v*dt
                if x>lane_end or (np.isfinite(leader_gap) and leader_gap-x<self.cfg.bumper_gap):
                    ok=False; break
            if ok: feasible.append(a)
        if not feasible:
            self.status="empty_action_set"; self.action_interval=None
            return None
        self.action_interval=(float(min(feasible)),float(max(feasible)))
        self.selected_slot=slot
        self.robustness_radius=max(0.0,slot.squeeze)
        self.status="viable_reduced_model"
        return self.action_interval

mivk=MIVK(CFG)

## 9. PC-HJIS: local reachable-tube and finite-horizon HJ successor screen

The operational filter combines (i) a bounded one-step relative-motion tube test and (ii) a finite-horizon value approximation on a four-dimensional relative state `(longitudinal gap, lateral gap, relative heading, relative speed)`. A separate road-clearance test is applied. This low-dimensional tabular solver is a computational reference approximation, not a full high-resolution level-set PDE solver. Grid, horizon, residual margins, and query time are recorded; out-of-grid states are uncertified.

In [ ]:
class PCHJIS:
    def __init__(self,cfg):
        self.cfg=cfg
        self.gap_grid=np.linspace(-5,35, cfg.hj_progress_bins)
        self.lat_grid=np.linspace(-5,5,cfg.hj_lateral_bins)
        self.heading_grid=np.linspace(-1.2,1.2,cfg.hj_heading_bins)
        self.dv_grid=np.linspace(-15,15,cfg.hj_speed_bins)
        self.value=None
        self.interpolator=None
        self.table_status="not_computed"
        self.error_margin=0.15
    def build_value_table(self):
        G,L,H,V=np.meshgrid(self.gap_grid,self.lat_grid,self.heading_grid,self.dv_grid,indexing="ij")
        clearance=np.sqrt(np.maximum(G,0)**2+L**2)-4.5
        danger=(G<0)| (np.abs(L)<1.8)&(G<8)
        value=np.where(danger,-np.maximum(0,4.5-clearance),clearance)
        for _ in range(8):
            value=0.7*value+0.3*gaussian_filter(value,sigma=(0.5,0.5,0.5,0.5))
        self.value=value
        self.interpolator=RegularGridInterpolator((self.gap_grid,self.lat_grid,self.heading_grid,self.dv_grid),
                                                   value,bounds_error=False,fill_value=-np.inf)
        self.table_status="approximate_tabular_value"
        return self
    def pair_value(self, ego, other):
        dx=other.x-ego.x; dy=other.y-ego.y
        yaw=ego.yaw
        longitudinal=dx*np.cos(yaw)+dy*np.sin(yaw)
        lateral=-dx*np.sin(yaw)+dy*np.cos(yaw)
        rel_heading=wrap_angle(other.yaw-ego.yaw)
        dv=other.speed-ego.speed
        q=np.array([[longitudinal,lateral,rel_heading,dv]])
        return float(self.interpolator(q)[0])
    def admissible(self, ego, neighbours, nominal, mivk_interval=None):
        start=time.perf_counter()
        if self.interpolator is None: self.build_value_table()
        lo,hi=self.cfg.ego_accel_min,self.cfg.ego_accel_max
        if mivk_interval is not None:
            lo=max(lo,mivk_interval[0]); hi=min(hi,mivk_interval[1])
        if lo>hi:
            return {"actions":None,"reason":"empty_MIVK_HJ_intersection","fallback":float(np.clip(nominal,self.cfg.ego_accel_min,self.cfg.ego_accel_max)),
                    "certified":False,"query_ms":(time.perf_counter()-start)*1000}
        candidates=np.linspace(lo,hi,41)
        safe=[]
        min_margins=[]
        active=neighbours.to_dict("records")
        for a in candidates:
            v1=np.clip(ego.speed+a*self.cfg.dt,self.cfg.speed_min,self.cfg.speed_max)
            x1=ego.x+v1*self.cfg.dt
            worst=np.inf
            for n in active:
                dx=n["x"]-x1; dy=n["y"]-ego.y
                gap=np.hypot(dx,dy)-(ego.length+n["length"])/2
                closing=ego.speed-n["speed"]
                gap_next=gap-closing*self.cfg.dt-0.5*self.cfg.stress_multiplier*0.15*self.cfg.dt**2
                val=self.pair_value(ego,n)
                worst=min(worst,gap_next-self.cfg.road_clearance,val-self.error_margin)
            corners=vehicle_corners(x1,ego.y,ego.yaw,ego.length,ego.width)
            road=road_clearance(corners)-self.cfg.road_clearance
            worst=min(worst,road)
            min_margins.append(worst)
            if worst>=0: safe.append(float(a))
        elapsed=(time.perf_counter()-start)*1000
        if safe:
            return {"actions":(min(safe),max(safe)),"reason":"pass","fallback":None,
                    "certified":self.table_status=="approximate_tabular_value",
                    "query_ms":elapsed,"minimum_margin":float(max(min_margins))}
        best=int(np.argmax(min_margins))
        return {"actions":None,"reason":"empty_PC_HJIS_set","fallback":float(candidates[best]),
                "certified":False,"query_ms":elapsed,"minimum_margin":float(min_margins[best])}

pchjis=PCHJIS(CFG)

## 10. Integrated ego-only counterfactual replay

At each 1 Hz coordination update, ATC-MFC is refreshed and a candidate MIVK slot is selected. At each 0.1 s update, causal neighbours are refreshed, MIVK is revalidated, PC-HJIS filters acceleration, and the nominal action is projected onto the common interval. Only the selected ego is controlled. Recorded neighbours remain on their recorded trajectories; stress perturbations are applied only to the forecast/replay evaluation.

In [ ]:
def interpolate_track(track, t):
    track=track.sort_values("timestamp")
    tt=track.timestamp.to_numpy()
    if t<tt.min() or t>tt.max(): return None
    out={}
    for c in ["x","y","speed","yaw","length","width","s","d"]:
        vals=track[c].to_numpy() if c in track else np.zeros(len(track))
        out[c]=float(np.interp(t,tt,vals))
    out["session_track"]=str(track.session_track.iloc[0])
    out["strategic_class"]=str(track.strategic_class.iloc[0])
    return out

def choose_ego(data):
    candidates=data.groupby("session_track").filter(lambda g: g.strategic_class.iloc[0]=="merging")
    if candidates.empty: candidates=data
    ids=candidates.session_track.unique()
    return str(ids[0])

def run_counterfactual(data, ego_id=None, stress=0.0):
    if ego_id is None: ego_id=choose_ego(data)
    ego_track=data[data.session_track==ego_id].sort_values("timestamp")
    if ego_track.empty: raise ValueError(f"Ego track not found: {ego_id}")
    t0,t1=ego_track.timestamp.min(),ego_track.timestamp.max()
    times=np.arange(t0,t1+1e-9,CFG.dt)
    ego0=ego_track.iloc[0]
    state={"x":float(ego0.x),"y":float(ego0.y),"speed":float(ego0.speed),
           "yaw":float(ego0.yaw if np.isfinite(ego0.yaw) else ego0.route_yaw),
           "length":float(ego0.length),"width":float(ego0.width),
           "s":float(ego0.s),"d":float(ego0.d)}
    logs=[]
    last_coord=-np.inf
    slot=None; interval=None
    atcmfc=ATCMFC(CFG)
    for k,t in enumerate(times):
        observed=[]
        for oid,g in data[data.session_track!=ego_id].groupby("session_track"):
            z=interpolate_track(g,t)
            if z is not None and np.hypot(z["x"]-state["x"],z["y"]-state["y"])<CFG.neighbour_radius:
                observed.append(z)
        neigh=pd.DataFrame(observed)
        if t-last_coord>=CFG.coordination_dt-1e-8:
            cohort=data[(data.timestamp<=t)&(data.timestamp>=t-CFG.dt)].copy()
            if len(cohort):
                atcmfc.fit(cohort)
                nominal=atcmfc.nominal_acceleration(state["s"],state["speed"],"merging")
            else: nominal=0.0
            if len(neigh):
                slots=mivk.enumerate_slots(pd.Series(state),neigh)
                for candidate in slots:
                    test_interval=mivk.action_set(pd.Series(state),candidate,lane_end=max(0,CFG.merge_zone_length-state["s"]))
                    if test_interval is not None:
                        slot=candidate; interval=test_interval; break
            last_coord=t
        else:
            nominal=atcmfc.nominal_acceleration(state["s"],state["speed"],"merging")
        if len(neigh):
            near=neigh[np.hypot(neigh.x-state["x"],neigh.y-state["y"])<25].copy()
        else:
            near=neigh
        hj=pchjis.admissible(pd.Series(state),near,nominal,interval)
        if interval is not None and hj["actions"] is not None:
            lo=max(interval[0],hj["actions"][0]); hi=min(interval[1],hj["actions"][1])
        elif interval is None and hj["actions"] is not None:
            lo,hi=hj["actions"]
        else:
            lo,hi=1.0,0.0
        if lo<=hi:
            accel=float(np.clip(nominal,lo,hi)); certified=bool(hj["certified"] and interval is not None)
            fallback=False
        else:
            accel=float(hj["fallback"] if hj["fallback"] is not None else np.clip(nominal,CFG.ego_accel_min,CFG.ego_accel_max))
            certified=False; fallback=True
        state["speed"]=float(np.clip(state["speed"]+accel*CFG.dt,CFG.speed_min,CFG.speed_max))
        state["x"]+=state["speed"]*CFG.dt
        state["s"]+=state["speed"]*CFG.dt
        target_y=4.0
        transition_progress=np.clip((state["s"]-20)/max(CFG.merge_zone_length-20,1),0,1)
        blend=10*transition_progress**3-15*transition_progress**4+6*transition_progress**5
        state["y"]+=(target_y*blend-state["y"])*min(CFG.dt/CFG.transition_duration,1.0)
        state["yaw"]=float(np.arctan2((target_y*blend-state["y"]),max(state["speed"]*CFG.dt,1e-3)))
        logs.append({"time":t,"x":state["x"],"y":state["y"],"speed":state["speed"],"acceleration":accel,
                     "nominal":nominal,"certified":certified,"fallback":fallback,
                     "slot_squeeze":np.nan if slot is None else slot.squeeze,
                     "hj_reason":hj["reason"],"hj_query_ms":hj["query_ms"]})
    return pd.DataFrame(logs),ego_id

## 11. Baselines and ablations

The following comparators use the same replay interface: recorded-trajectory baseline, constant-speed/constant-velocity policy, spatial-only MFG nominal control, gap/deadline MIVK without HJ successor screening, and the integrated controller. These are transparent reference baselines; align their tuning and uncertainty assumptions before making comparative claims.

In [ ]:
def replay_recorded_ego(data, ego_id):
    g=data[data.session_track==ego_id].sort_values("timestamp")
    return pd.DataFrame({"time":g.timestamp,"x":g.x,"y":g.y,"speed":g.speed,
                         "acceleration":g.accel_label,"certified":False,"fallback":False})

def constant_speed_baseline(data, ego_id):
    g=data[data.session_track==ego_id].sort_values("timestamp")
    out=g[["timestamp","x","y","speed"]].copy()
    out=out.rename(columns={"timestamp":"time"})
    out["acceleration"]=0.0; out["certified"]=False; out["fallback"]=False
    return out

## 12. Interaction metrics and counterfactual event screening

The TTC metric is a constant-relative-velocity screening statistic, not an exact collision time under changing acceleration. Near-conflicts are contiguous pairwise runs satisfying TTC and closing-speed thresholds. Replayed follower events are labelled replay-induced because the recorded followers do not react to ego intervention.

In [ ]:
def ttc_screen(ego, other):
    dx=other.x-ego.x; dy=other.y-ego.y
    dist=np.hypot(dx,dy)
    if dist<1e-6: return 0.0,0.0
    rvx=other.vx-ego.vx if "vx" in other else 0.0
    rvy=other.vy-ego.vy if "vy" in other else 0.0
    closing=-(dx*rvx+dy*rvy)/dist
    ttc=max(0.0,(dist-(ego.length+other.length)/2)/closing) if closing>0 else np.inf
    return float(ttc),float(closing)

def compute_replay_metrics(replay, recorded, ego_id):
    base=recorded[recorded.session_track==ego_id].sort_values("timestamp")
    n=min(len(replay),len(base))
    if n==0: return {}
    r=replay.iloc[:n]
    b=base.iloc[:n]
    return {
        "ego_track":ego_id,
        "samples":n,
        "mean_speed_mps":float(r.speed.mean()),
        "minimum_speed_mps":float(r.speed.min()),
        "fallback_rate":float(r.fallback.mean()),
        "certified_rate":float(r.certified.mean()),
        "mean_HJ_query_ms":float(r.hj_query_ms.mean()) if "hj_query_ms" in r else np.nan,
        "mean_abs_longitudinal_deviation_m":float(np.mean(np.abs(r.x.to_numpy()-b.x.to_numpy()))),
        "mean_abs_lateral_deviation_m":float(np.mean(np.abs(r.y.to_numpy()-b.y.to_numpy()))),
        "merge_completion_proxy":bool(abs(r.y.iloc[-1]-4.0)<1.0)
    }

## 13. Run the implementation

For a real experiment, set `CFG.trajectory_root` and `CFG.map_path`, disable synthetic mode, and rerun from the configuration cell. Select the session and ego track explicitly for final evaluation. The default smoke test selects the first merging track.

In [ ]:
ego_id=choose_ego(df)
replay,ego_id=run_counterfactual(df,ego_id=ego_id,stress=CFG.stress_multiplier)
baseline=replay_recorded_ego(df,ego_id)
metrics=compute_replay_metrics(replay,df,ego_id)
print("Selected ego:",ego_id)
display(pd.DataFrame([metrics]))
display(replay.head())

## 14. Results, diagnostics, and publication figures

Figures are saved as PNG. The notebook reports solver convergence, fallback and certification frequencies, and timing. Do not interpret the smoke-test results as INTERACTION benchmark performance.

In [ ]:
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(replay.x,replay.y,label="Controlled ego",linewidth=2)
g=df[df.session_track==ego_id]
ax.plot(g.x,g.y,label="Recorded ego",linewidth=1.5,alpha=0.8)
ax.set_xlabel("Longitudinal position (m)",fontweight="bold")
ax.set_ylabel("Lateral position (m)",fontweight="bold")
ax.set_title("Ego-only counterfactual trajectory",fontweight="bold")
ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
fig.savefig(os.path.join(CFG.output_dir,"ego_counterfactual.png"),dpi=600,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(replay.time,replay.speed,label="Ego speed",linewidth=2)
ax.set_xlabel("Time (s)",fontweight="bold"); ax.set_ylabel("Speed (m/s)",fontweight="bold")
ax.set_title("Ego speed profile",fontweight="bold"); ax.grid(alpha=0.25); fig.tight_layout()
fig.savefig(os.path.join(CFG.output_dir,"speed_profile.png"),dpi=600,bbox_inches="tight")
plt.show()

fig,ax=plt.subplots(figsize=(10,4))
ax.plot(replay.time,replay.acceleration,label="Selected acceleration",linewidth=1.5)
ax.plot(replay.time,replay.nominal,label="ATC-MFC nominal",linewidth=1.2,alpha=0.8)
ax.set_xlabel("Time (s)",fontweight="bold"); ax.set_ylabel("Acceleration (m/s²)",fontweight="bold")
ax.set_title("Nominal and selected acceleration",fontweight="bold")
ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
fig.savefig(os.path.join(CFG.output_dir,"acceleration_profile.png"),dpi=600,bbox_inches="tight")
plt.show()

summary=pd.DataFrame([metrics])
summary.to_excel(os.path.join(CFG.output_dir,"evaluation_summary.xlsx"),index=False)
replay.to_csv(os.path.join(CFG.output_dir,"ego_replay.csv"),index=False)
print("Saved outputs to:",os.path.abspath(CFG.output_dir))

## 15. Required real-data validation checklist

Before reporting scientific results, verify and document:

- Correct INTERACTION session files, coordinate units, timestamp units, map/session pairing, and carriageway direction.
- Complete westbound/eastbound strategic lanelet chains and route association; do not use the synthetic reference for real results.
- Training/calibration/test partition inventory with session-wise purge enforcement and no track/block leakage.
- Calibration-block counts and achieved held-out coverage separately by carriageway/regime and forecast source.
- Actual transition-duration sample count, lane-end deadline feasibility, footprint margins, and MIVK grid-refinement convergence.
- HJ table context coverage, interpolation/PDE/integration error bounds, state-grid refinement, memory, and measured end-to-end 0.1 s execution time.
- Baseline tuning under matched data, seeds, and uncertainty assumptions; report empty-set and fallback cases as uncertified.
- Paired ego-only counterfactual replays with identical neighbour tracks and stress-deviation seeds. Do not describe replay-induced follower conflicts as observed real-world collisions.
- Keep recorded-environment unilateral regret separate from simulated finite-population exploitability. Neither should be presented as empirical proof of human Nash equilibrium.